<a href="https://colab.research.google.com/github/kayeneii/Floodgate/blob/main/floodgate_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mapping Flood Risk Zones in Lagos Using Open Data
## *Step 1: Data Collection and Preprocessing*

- Authenticate: Earth Engine & Google Drive.

- Define study area and date window.

- Query CHIRPS and check collection size — stop with a clear message if 0.

- Aggregate (sum) and reproject to DEM projection.

- Export images to Drive using ee.batch.Export.image.toDrive() and start the tasks

- Poll / print task.status() to know how it started.

In [ ]:
# Colab setup: install required Python packages
!pip install earthengine-api geemap rioxarray rasterio geopandas osmnx folium contextily shapely pyproj -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.4 MB/s eta 0:00:00


In [36]:
# Authenticate EE and mount Google Drive
import ee, time, os
ee.Authenticate()   # follow interactive prompt
ee.Initialize(project="thinking-league-472119-r1")

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
# PARAMETERS - edit these
minLon, minLat, maxLon, maxLat = 3.0, 6.0, 4.0, 7.0
start_date = '2025-04-01'
end_date   = '2025-06-30'
export_scale = 30  # meters
drive_folder = 'GEE_exports'  # will be created in your Drive root

In [49]:
# Build study geometry
study = ee.Geometry.Rectangle([minLon, minLat, maxLon, maxLat])

In [52]:
# Build CHIRPS collection and safe-check size
chirps_col = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(start_date, end_date).filterBounds(study)

In [53]:
# IMPORTANT: get size server-side and then client-side
chirps_count = chirps_col.size().getInfo()  # safe in Colab, but will block briefly
print('CHIRPS count for your filters =', chirps_count)
if chirps_count == 0:
    raise SystemExit('ERROR: No CHIRPS images found for that date range and region. Try different dates (CHIRPS available from 1981-01-01) or expand the region.')

CHIRPS count for your filters = 90


In [54]:
# Aggregate to seasonal total
chirps_total = chirps_col.sum().clip(study).toFloat()

In [55]:
# DEM
dem = ee.Image('USGS/SRTMGL1_003').clip(study).toFloat()
slope = ee.Terrain.slope(dem)

In [56]:
# Ensure rainfall image projection matches DEM (reproject to DEM's projection)
chirps_total = chirps_total.reproject(crs=dem.projection(), scale=export_scale)

In [57]:
# === Export functions with robust start + status printing ===
def export_image_to_drive(image, description, file_name_prefix, folder, region, scale, maxPixels=1e10):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        region=region,
        scale=scale,
        maxPixels=maxPixels
    )
    task.start()
    print(f'Started task {description} -> {folder}/{file_name_prefix}. Monitoring status...')
    # Print status for a few seconds
    for i in range(30):  # polls for ~30*2s = 60s
        status = task.status()
        print(i, status['state'])
        if status['state'] in ('COMPLETED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    print('Final status:', task.status())
    return task

In [58]:
# Start exports (small bbox for testing)
t1 = export_image_to_drive(chirps_total, 'chirps_seasonal_COLAB_TEST', 'chirps_seasonal_COLAB_TEST', drive_folder, study, export_scale)
t2 = export_image_to_drive(dem, 'dem_sample_COLAB_TEST', 'dem_sample_COLAB_TEST', drive_folder, study, export_scale)
t3 = export_image_to_drive(slope, 'slope_sample_COLAB_TEST', 'slope_sample_COLAB_TEST', drive_folder, study, export_scale)

print('Export tasks submitted. Open https://drive.google.com/drive/my-drive and look for the folder:', drive_folder)

Started task chirps_seasonal_COLAB_TEST -> GEE_exports/chirps_seasonal_COLAB_TEST. Monitoring status...
0 READY
1 READY
2 READY
3 READY
4 READY
5 READY
6 READY
7 READY
8 READY
9 READY
10 READY
11 READY
12 READY
13 READY
14 READY
15 READY
16 READY
17 READY
18 READY
19 READY
20 READY
21 READY
22 READY
23 READY
24 READY
25 READY
26 READY
27 READY
28 READY
29 READY
Final status: {'state': 'READY', 'description': 'chirps_seasonal_COLAB_TEST', 'priority': 100, 'creation_timestamp_ms': 1760991000671, 'update_timestamp_ms': 1760991007746, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'YFM3HGX4VV4AOODXIOXP6SYL', 'name': 'projects/thinking-league-472119-r1/operations/YFM3HGX4VV4AOODXIOXP6SYL'}
Started task dem_sample_COLAB_TEST -> GEE_exports/dem_sample_COLAB_TEST. Monitoring status...
0 READY
1 READY
2 READY
3 READY
4 READY
5 READY
6 READY
7 READY
8 READY
9 READY
10 READY
11 READY
12 READY
13 READY
14 READY
15 READY
16 READY
17 READY
18 READY
19 READY
20 READY
21 READY
22 READY
23